# Integrated Practice: Copyright Case Law RAG  
- 종합실습: 저작권 판례 기반 RAG 시스템 구축

### [Preparation]
- Dataset Upload
- Data Inspection

### [Knowledge Base / Indexing]
- Step1. Document Loader — Excel → LangChain Document
- Step2. Text Splitter — 판례 본문 Chunking
- Step3. Embedding — Chunk → Vector
- Step4. Vector Store — ChromaDB

### [Runtime]
- Step5. Retriever
- Step6. Prompt Construction
- Step7. LLM
- Step8. Chain & Pipeline

### [Note]
- 이 코드는 Baseline입니다.
- Day2, Day3에서 학습할 Chunking, Retrieval, Query Transformation, Reranking 등의 기법을 적용해 품질을 개선합니다.

In [ ]:
!pip install -q langchain langchain-openai langchain-community langchain-chroma chromadb openpyxl pandas rank_bm25


## Preparation 1. API Key 설정

In [ ]:
import os
from google.colab import userdata

# Colab 왼쪽의 열쇠 아이콘(Secrets)에 OPENAI_API_KEY 등록 필요
api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Colab Secrets에 OPENAI_API_KEY를 등록해 주세요.")
os.environ["OPENAI_API_KEY"] = api_key

print("API Key 설정 완료")

## Preparation 2. 판례 데이터와 Evaluation Set 업로드

공개 저장소에는 원본 Excel 파일을 포함하지 않습니다. 아래 셀을 실행한 뒤 본인이 사용할 수 있는 두 파일을 직접 업로드합니다.

- `Copyright_case_law_Supreme.xlsx`
- `Copyright_RAG_Evaluation.xlsx`


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

DATA_PATH = '/content/Copyright_case_law_Supreme.xlsx'
EVAL_PATH = '/content/Copyright_RAG_Evaluation.xlsx'

required_files = {Path(DATA_PATH).name, Path(EVAL_PATH).name}
missing = required_files - set(uploaded)
if missing:
    raise ValueError(f'필요한 파일이 없습니다: {sorted(missing)}')

print('판례 데이터와 Evaluation Set 업로드 완료')


## Preparation 3. 판례 데이터와 Evaluation Set 확인


In [ ]:
import pandas as pd

df = pd.read_excel(DATA_PATH)
eval_df = pd.read_excel(EVAL_PATH)

required_case_columns = {"사건명", "사건번호", "선고일자", "법원명", "판례내용"}
required_eval_columns = {"ID", "Category", "Question"}

if not required_case_columns.issubset(df.columns):
    raise ValueError(f"판례 데이터에 필요한 열이 없습니다: {required_case_columns - set(df.columns)}")
if not required_eval_columns.issubset(eval_df.columns):
    raise ValueError(f"Evaluation Set에 필요한 열이 없습니다: {required_eval_columns - set(eval_df.columns)}")

print(f"판례 수: {len(df):,}")
print(f"판례 컬럼: {df.columns.tolist()}")
print(f"Evaluation Question 수: {len(eval_df):,}")
display(df.head(3))
display(eval_df)


# Indexing : Knowledge Base 구축

## Step1. Document Loader

Excel의 각 판례 행을 Document 객체로 변환

- `page_content` : 판례 본문
- `metadata` : 사건명, 사건번호, 선고일자, 법원명

Metadata를 유지하면 이후 검색 결과에서 어떤 판례가 검색되었는지 추적할 수 있음


In [ ]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():
    content = str(row["판례내용"])

    metadata = {
        "case_name": str(row["사건명"]),
        "case_no": str(row["사건번호"]),
        "decision_date": str(row["선고일자"]),
        "court": str(row["법원명"]),
    }

    documents.append(
        Document(
            page_content=content,
            metadata=metadata
        )
    )

print(f"Documents: {len(documents):,}")
print("\n[Metadata Sample]")
print(documents[0].metadata)
print("\n[Content Sample]")
print(documents[0].page_content[:500])


## Step2. Text Splitter — Baseline Chunking

판례 본문 Chunk Baseline 설정으로 시작

- `chunk_size = 1000`
- `chunk_overlap = 150`
  
> Chunk Size / Overlap / 구조 기반 Chunking 등을 개선 대상으로 실험할 수 있음


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
)

docs = text_splitter.split_documents(documents)

print(f"Original Documents: {len(documents):,}")
print(f"Chunks: {len(docs):,}")

print("\n[Chunk Sample]")
print(docs[0].page_content[:500])

print("\n[Metadata after Chunking]")
print(docs[0].metadata)


## Step3. Embedding

판례 Chunk를 의미를 표현하는 Vector로 변환

- Model: `text-embedding-3-small`


In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("Embedding Model 설정 완료")


## Step4. Vector Store — ChromaDB

별도의 `persist_directory`를 사용하지 않고 현재 Colab 런타임에서만 사용하는 ChromaDB를 생성

셀을 다시 실행할 때 기존 Collection과 충돌하거나 문서가 중복 적재되지 않도록  
실행할 때마다 새로운 Collection 이름을 생성

> Colab 런타임이 종료시 이 Vector DB도 함께 사라짐


In [ ]:
import uuid
from langchain_chroma import Chroma

# 셀을 다시 실행할 때마다 새로운 Collection 생성(중복적재 방지)
collection_name = f"copyright_baseline_{uuid.uuid4().hex[:8]}"

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name=collection_name,
)

print("ChromaDB 생성 완료")
print(f"Collection: {collection_name}")
print(f"저장된 벡터 수: {vectorstore._collection.count():,}")


# Runtime

## Step5. Retriever

Baseline에서는 Vector Similarity Search + Top-k 3 사용


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

print("Retriever 설정 완료: similarity / top-k = 3")


### Retrieval 결과 확인하기

In [ ]:
query = "온라인 게임의 규칙이나 전개 방식이 비슷하면 저작권 침해인가요?"

retrieved_docs = retriever.invoke(query)

print(f"질문: {query}")
print(f"\n검색된 Chunk 수: {len(retrieved_docs)}")

for i, doc in enumerate(retrieved_docs, 1):
    print("=" * 80)
    print(
        f"[{i}] {doc.metadata.get('case_name')} "
        f"/ {doc.metadata.get('case_no')} "
        f"/ {doc.metadata.get('decision_date')}"
    )
    print(doc.page_content[:800])


## Step6. Prompt Construction

검색된 판례 Chunk를 Context로 넣고, 질문과 답변 지침을 함께 구성


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
당신은 저작권 판례를 분석하는 AI Assistant입니다.

[Instruction]
아래 [참고 판례]만을 근거로 질문에 답변하세요.
근거가 충분하지 않으면 추측하지 말고
"제공된 판례만으로는 충분한 근거를 찾기 어렵습니다."라고 답하세요.

답변에는 핵심 판단과 그 근거를 간결하게 설명하세요.

[Context]
{context}

[Question]
{question}

[Answer]
""")

def format_docs(docs):
    return "\n\n".join(
        f"[사건명: {d.metadata.get('case_name')} | "
        f"사건번호: {d.metadata.get('case_no')} | "
        f"선고일자: {d.metadata.get('decision_date')}]\n"
        f"{d.page_content}"
        for d in docs
    )

print("Prompt Template 구성 완료")


### What Does the LLM Actually See?

Retriever가 가져온 판례 Chunk가 실제 Prompt의 Context에 어떻게 들어가는지 확인


In [ ]:
final_prompt = prompt.invoke({
    "context": format_docs(retrieved_docs),
    "question": query,
})

print(final_prompt.messages[0].content)


## Step7. LLM

검색된 Context를 바탕으로 답변을 생성할 LLM을 설정


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.2",
    temperature=0.2,
)

print("LLM 설정 완료")


## Step8. Chain & Pipeline

Retriever → Prompt → LLM → Output Parser를 하나의 실행 흐름으로 연결


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("Baseline RAG Chain 구성 완료")


## Baseline RAG 테스트

In [ ]:
question = "제품 사진도 모두 저작권으로 보호받을 수 있나요?"

answer = rag_chain.invoke(question)

print("[Question]")
print(question)
print("\n[Answer]")
print(answer)


## 응답과 Retrieval 근거 함께 확인하기

답변 품질을 볼 때는 최종 답변만 보지 말고 Retrieval 결과가 적절했는지 먼저 확인


In [ ]:
def ask_rag(question):
    retrieved = retriever.invoke(question)
    answer = rag_chain.invoke(question)

    print("[Question]")
    print(question)

    print("\n[Answer]")
    print(answer)

    print("\n[Retrieved Cases]")
    for i, doc in enumerate(retrieved, 1):
        print(
            f"{i}. {doc.metadata.get('case_name')} "
            f"({doc.metadata.get('case_no')}, {doc.metadata.get('decision_date')})"
        )

    return {
        "question": question,
        "answer": answer,
        "retrieved_docs": retrieved,
    }

result = ask_rag(
    "저작자가 동의한 범위 안에서 저작물을 수정해도 문제가 되나요?"
)


# [Task Guide: Copyright Case Law RAG]

## Baseline Observation — Limits of Basic RAG

먼저 제공된 Baseline RAG의 한계를 확인합니다.

- 관련 판례가 Top-k 안에 검색되는가?
- 관련 판례가 검색되었지만 필요한 판단 부분이 Chunk 경계에서 잘리지는 않는가?
- 서로 다른 사건의 Chunk가 섞여 답변이 모호해지지는 않는가?
- 질문 표현과 판례 표현이 다른 경우에도 잘 검색되는가?
- 검색은 잘 되었지만 최종 답변 품질이 충분하지 않은 경우가 있는가?

## Challenge — Improve the Baseline

Day2 / Day3에서 학습한 Advanced RAG Technique 중
2개 이상을 적용하여 Baseline의 검색 및 응답 품질을 개선합니다.

예시:
- Chunking Strategy
- Query Transformation / Multi Query
- Hybrid Search
- Metadata Filtering
- Reranking
- Verification / Retry
- ...

단순히 기법을 적용하는 것보다
Baseline의 어떤 문제를 해결하기 위해 해당 기법을 선택했는지 설명하는 것이 중요합니다.

## Evaluation — Did It Really Improve?

동일한 Evaluation Question Set을 사용하여
Baseline과 개선 모델의 결과를 비교합니다.

다음 내용을 중심으로 확인하세요.

- Retrieval 결과는 개선되었는가?
- 관련 판례가 더 정확하게 검색되는가?
- 최종 답변의 정확성 및 근거성이 개선되었는가?
- 적용한 Technique이 모든 질문에 동일하게 효과적이었는가?
- 개선되지 않았거나 오히려 악화된 경우는 무엇인가?

가능한 경우 RAGAS 등의 평가 방법을 활용하여
정량적인 결과도 함께 비교해 보세요.

## Final Workflow

> Baseline  
> → Failure Analysis  
> → Advanced RAG Technique 적용  
> → Before / After Comparison  
> → Evaluation  
> → Lessons Learned


# Advanced RAG 개선 실습

## 설계 원칙

Baseline과 Advanced 모델은 같은 판례 데이터, 같은 임베딩 모델, 같은 LLM, 같은 Evaluation Question을 사용합니다. 검색 파이프라인만 바꾸어 개선 효과를 비교합니다.

| 구분 | 적용 기법 | 선택 이유 |
|---|---|---|
| Day 2 | 판례 구조 기반 Chunking | 판결 이유와 번호 문단이 Chunk 경계에서 잘리는 문제를 줄이기 위해 적용 |
| Day 2 | BM25 + Dense MMR Hybrid Retrieval | 법률 용어의 정확한 일치와 질문의 의미적 유사성을 함께 반영하기 위해 적용 |
| Day 2 | Reciprocal Rank Fusion(RRF) | BM25와 Dense 검색 순위를 한쪽 점수 체계에 종속되지 않고 결합하기 위해 적용 |
| Day 3 | Evidence Verification + Retry | 검색 근거가 부족하면 수정된 검색어로 한 번 더 검색하기 위해 적용 |

> 이 노트북은 **LangGraph를 사용하지 않습니다.** 일반 Python 함수와 LangChain Runnable만 사용합니다.


## Technique 1. 판례 구조 기반 Chunking

Baseline은 1,000자 단위로만 분할합니다. Advanced는 `【이 유】`, 번호 문단, 항목 구분을 우선 고려하고 Chunk를 조금 더 작게 만들어 핵심 법리가 한 Chunk 안에 들어올 가능성을 높입니다.


In [ ]:
import re
import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

legal_separators = [
    "\n【이 유】", "\n【이유】", "\n이 유", "\n이유",
    "\n1. ", "\n2. ", "\n3. ", "\n4. ", "\n5. ",
    "\n가. ", "\n나. ", "\n다. ", "\n라. ",
    "\n(1) ", "\n(2) ", "\n(3) ", "\n(4) ",
    "\n\n", "\n", ". ", " ", ""
]

advanced_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120,
    separators=legal_separators,
    length_function=len,
)

advanced_docs = advanced_splitter.split_documents(documents)

for chunk_index, doc in enumerate(advanced_docs):
    doc.metadata["chunk_id"] = f"advanced-{chunk_index}"
    header = (
        f"[사건명: {doc.metadata.get('case_name')} | "
        f"사건번호: {doc.metadata.get('case_no')} | "
        f"선고일자: {doc.metadata.get('decision_date')}]\n"
    )
    doc.page_content = header + doc.page_content

advanced_collection_name = f"copyright_advanced_{uuid.uuid4().hex[:8]}"
advanced_vectorstore = Chroma.from_documents(
    documents=advanced_docs,
    embedding=embedding_model,
    collection_name=advanced_collection_name,
)

print(f"Baseline Chunks: {len(docs):,}")
print(f"Advanced Chunks: {len(advanced_docs):,}")
print(f"Advanced Collection: {advanced_collection_name}")


## Technique 2. Hybrid Retrieval + MMR + RRF

- **BM25**: `동일성유지권`, `서체도안`처럼 정확한 법률 용어 검색에 강합니다.
- **Dense MMR**: 질문과 판례의 표현이 달라도 의미가 비슷한 문서를 찾고, 중복 Chunk를 줄입니다.
- **RRF**: 두 검색 결과의 순위를 합쳐 최종 Top-k를 결정합니다.


In [ ]:
from collections import defaultdict
from langchain_community.retrievers import BM25Retriever

def korean_tokenize(text):
    # 조사까지 완벽하게 분석하는 형태소 분석기는 아니지만,
    # 한글/영문/숫자 단위로 나누어 BM25에 사용할 수 있습니다.
    return re.findall(r"[가-힣A-Za-z0-9]+", text.lower())

bm25_retriever = BM25Retriever.from_documents(
    advanced_docs,
    preprocess_func=korean_tokenize,
)
bm25_retriever.k = 8

dense_mmr_retriever = advanced_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 30,
        "lambda_mult": 0.7,
    },
)

def document_key(doc):
    return doc.metadata.get("chunk_id") or (
        doc.metadata.get("case_no"), doc.page_content[:100]
    )

def rrf_merge(ranked_lists, top_k=3, rrf_k=60):
    scores = defaultdict(float)
    documents_by_key = {}

    for ranked_docs in ranked_lists:
        for rank, doc in enumerate(ranked_docs, start=1):
            key = document_key(doc)
            scores[key] += 1.0 / (rrf_k + rank)
            documents_by_key[key] = doc

    ranked_keys = sorted(scores, key=scores.get, reverse=True)

    # 같은 사건만 반복되지 않도록 사건별 최대 2개 Chunk로 제한합니다.
    selected = []
    case_counts = defaultdict(int)
    for key in ranked_keys:
        doc = documents_by_key[key]
        case_no = str(doc.metadata.get("case_no"))
        if case_counts[case_no] >= 2:
            continue
        selected.append(doc)
        case_counts[case_no] += 1
        if len(selected) == top_k:
            break

    return selected

def hybrid_retrieve(question, top_k=3):
    dense_docs = dense_mmr_retriever.invoke(question)
    bm25_docs = bm25_retriever.invoke(question)
    return rrf_merge([dense_docs, bm25_docs], top_k=top_k)

sample_hybrid_docs = hybrid_retrieve(
    "온라인 게임의 규칙이나 전개 방식이 비슷하면 저작권 침해인가요?",
    top_k=3,
)

for i, doc in enumerate(sample_hybrid_docs, 1):
    print(f"{i}. {doc.metadata.get('case_name')} ({doc.metadata.get('case_no')})")


## Technique 3. Evidence Verification + Retry

첫 검색 결과에 질문에 직접 답할 근거가 있는지 검증합니다. 근거가 부족할 때만 검증 모델이 만든 법률 검색어로 **한 번** 다시 검색합니다. 무한 반복은 하지 않습니다.

Structured Output을 사용해 검증 결과를 `충분/부족`, `이유`, `재검색어`로 받습니다.


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class VerificationResult(BaseModel):
    sufficient: bool = Field(description="질문에 직접 답할 판례 근거가 충분한지 여부")
    reason: str = Field(description="판단 이유를 한 문장으로 설명")
    retry_query: str = Field(description="근거가 부족할 때 사용할 법률 중심 재검색어. 충분하면 빈 문자열")

verification_prompt = ChatPromptTemplate.from_template("""
당신은 저작권 판례 RAG의 검색 결과 검증기입니다.

[판단 기준]
1. 검색 결과 중 적어도 하나가 질문의 핵심 법률 쟁점에 직접 답해야 합니다.
2. 사건명만 비슷하고 법률 쟁점이 다르면 부족한 근거입니다.
3. 근거가 부족하면 판례에서 사용할 법률 용어 중심으로 재검색어를 작성합니다.

[Question]
{question}

[Retrieved Evidence]
{context}
""")

verification_chain = verification_prompt | llm.with_structured_output(VerificationResult)

def verify_evidence(question, retrieved_docs):
    compact_context = format_docs(retrieved_docs)
    return verification_chain.invoke({
        "question": question,
        "context": compact_context,
    })

def advanced_retrieve(question, top_k=3):
    first_docs = hybrid_retrieve(question, top_k=6)
    verdict = verify_evidence(question, first_docs[:3])

    if verdict.sufficient:
        return first_docs[:top_k], verdict, False

    retry_query = verdict.retry_query.strip() or question
    retry_docs = hybrid_retrieve(retry_query, top_k=6)
    final_docs = rrf_merge([first_docs, retry_docs], top_k=top_k)
    return final_docs, verdict, True

advanced_answer_chain = prompt | llm | StrOutputParser()

def answer_with_docs(question, retrieved_docs):
    return advanced_answer_chain.invoke({
        "context": format_docs(retrieved_docs),
        "question": question,
    })

print("Verification + Retry 구성 완료")
print("LangGraph 사용 여부: False")


# Evaluation

## 동일 Evaluation Set과 정답 판례

12개 질문 중 쟁점이 서로 다른 6개를 선택했습니다. Baseline과 Advanced에 **완전히 같은 6개 질문**을 입력합니다.

Retrieval 평가는 정답 판례가 Top-3 안에 있는지 확인하고, Answer Quality는 같은 LLM Judge가 두 답변을 동시에 비교합니다.


In [ ]:
EVAL_IDS = ["E01", "E05", "E07", "E10", "E11", "E12"]

ground_truth = {
    "E01": {
        "case_no": "99다50552",
        "reference": "서체도안 자체와 서체파일 프로그램의 보호 여부를 구분해야 하며, 유사하다는 사실만으로 손해배상책임이 자동 성립하지 않는다."
    },
    "E05": {
        "case_no": "98다43366",
        "reference": "제품 사진은 항상 보호되는 것이 아니며 피사체 선정, 구도, 빛, 각도 등에서 촬영자의 개성과 창조성이 인정되어야 한다."
    },
    "E07": {
        "case_no": "2017다212095",
        "reference": "게임의 장르나 아이디어, 일반적인 규칙이 비슷하다는 이유만으로 곧바로 침해가 되지는 않지만 구성요소의 창작적인 선택·배열·조합이 실질적으로 유사하면 침해가 인정될 수 있다."
    },
    "E10": {
        "case_no": "2010다79923",
        "reference": "저작자가 명시적 또는 묵시적으로 동의한 범위 안의 변경은 동일성유지권 침해가 아니며, 동의 범위는 계약과 이용 상황 등을 종합하여 판단한다."
    },
    "E11": {
        "case_no": "2012도13748",
        "reference": "링크는 일반적으로 저작물의 위치나 경로를 나타낼 뿐이어서 제공만으로 복제·전송의 직접 침해가 바로 성립하는 것은 아니다. 구체적 방조책임은 링크 대상과 행위 태양을 별도로 살펴야 한다."
    },
    "E12": {
        "case_no": "94누5632",
        "reference": "등록관청은 명백한 비저작물 여부 같은 형식적 요건은 심사할 수 있지만 독창성의 정도, 보호 범위, 권리 귀속 같은 실체적 권리관계까지 심사할 권한은 없다."
    },
}

evaluation_set = eval_df[eval_df["ID"].isin(EVAL_IDS)].copy()
evaluation_set["Expected_Case_No"] = evaluation_set["ID"].map(
    lambda x: ground_truth[x]["case_no"]
)

display(evaluation_set)


## 평가 함수

- **Hit@3**: 정답 판례가 검색 결과 3개 안에 있으면 1
- **MRR@3**: 정답 판례가 위쪽에 검색될수록 높은 점수
- **Correctness**: 기준 답안과 비교한 정확성(1~5)
- **Groundedness**: 검색된 판례에 근거한 답변인지(1~5)


In [ ]:
class ComparisonScore(BaseModel):
    baseline_correctness: int = Field(ge=1, le=5)
    baseline_groundedness: int = Field(ge=1, le=5)
    advanced_correctness: int = Field(ge=1, le=5)
    advanced_groundedness: int = Field(ge=1, le=5)
    reason: str

comparison_prompt = ChatPromptTemplate.from_template("""
당신은 저작권 판례 RAG 답변 평가자입니다.
같은 질문에 대한 Baseline과 Advanced 답변을 동일한 기준으로 평가하세요.

[평가 기준]
- Correctness: 기준 답안의 핵심 법리를 정확히 포함하는가? (1~5)
- Groundedness: 각 답변이 제공된 검색 근거에서 확인되는 내용만 말하는가? (1~5)
- 두 답변의 문체가 아니라 법률 내용과 근거성을 평가하세요.

[Question]
{question}

[Reference Answer]
{reference}

[Baseline Evidence]
{baseline_context}

[Baseline Answer]
{baseline_answer}

[Advanced Evidence]
{advanced_context}

[Advanced Answer]
{advanced_answer}
""")

comparison_chain = comparison_prompt | llm.with_structured_output(ComparisonScore)

def normalize_case_no(value):
    return re.sub(r"[^0-9가-힣]", "", str(value))

def case_rank(retrieved_docs, expected_case_no):
    expected = normalize_case_no(expected_case_no)
    for rank, doc in enumerate(retrieved_docs, start=1):
        current = normalize_case_no(doc.metadata.get("case_no", ""))
        if expected and expected in current:
            return rank
    return None

def case_list(retrieved_docs):
    return [str(doc.metadata.get("case_no")) for doc in retrieved_docs]

print("평가 함수 준비 완료")


## Baseline vs Advanced 실행

아래 셀이 실습의 핵심 실행 셀입니다. 질문별로 Baseline과 Advanced를 차례로 실행하고 같은 기준으로 비교합니다.


In [ ]:
evaluation_rows = []
answer_details = []

for _, item in evaluation_set.iterrows():
    eval_id = item["ID"]
    question = item["Question"]
    expected_case = ground_truth[eval_id]["case_no"]
    reference = ground_truth[eval_id]["reference"]

    print(f"평가 중: {eval_id} - {item['Category']}")

    # 1) Baseline: similarity / Top-3
    baseline_docs = retriever.invoke(question)
    baseline_answer = rag_chain.invoke(question)

    # 2) Advanced: structure-aware chunks + hybrid + verification/retry / Top-3
    advanced_docs_result, verification, retried = advanced_retrieve(
        question, top_k=3
    )
    advanced_answer = answer_with_docs(question, advanced_docs_result)

    # 3) 동일한 Judge가 두 답변을 한 번에 평가
    score = comparison_chain.invoke({
        "question": question,
        "reference": reference,
        "baseline_context": format_docs(baseline_docs),
        "baseline_answer": baseline_answer,
        "advanced_context": format_docs(advanced_docs_result),
        "advanced_answer": advanced_answer,
    })

    baseline_rank = case_rank(baseline_docs, expected_case)
    advanced_rank = case_rank(advanced_docs_result, expected_case)

    evaluation_rows.append({
        "ID": eval_id,
        "Category": item["Category"],
        "Expected_Case": expected_case,
        "Baseline_Cases": case_list(baseline_docs),
        "Advanced_Cases": case_list(advanced_docs_result),
        "Baseline_Hit@3": int(baseline_rank is not None),
        "Advanced_Hit@3": int(advanced_rank is not None),
        "Baseline_MRR@3": 0 if baseline_rank is None else 1 / baseline_rank,
        "Advanced_MRR@3": 0 if advanced_rank is None else 1 / advanced_rank,
        "Baseline_Correctness": score.baseline_correctness,
        "Advanced_Correctness": score.advanced_correctness,
        "Baseline_Groundedness": score.baseline_groundedness,
        "Advanced_Groundedness": score.advanced_groundedness,
        "Verification_Retry": retried,
        "Verification_Reason": verification.reason,
        "Judge_Reason": score.reason,
    })

    answer_details.append({
        "ID": eval_id,
        "Question": question,
        "Baseline_Answer": baseline_answer,
        "Advanced_Answer": advanced_answer,
    })

evaluation_results = pd.DataFrame(evaluation_rows)
answer_details_df = pd.DataFrame(answer_details)

display(evaluation_results)


## 정량 비교와 대표 답변 확인


In [ ]:
import matplotlib.pyplot as plt

summary_table = pd.DataFrame({
    "Metric": ["Hit@3", "MRR@3", "Correctness (1~5)", "Groundedness (1~5)"],
    "Baseline": [
        evaluation_results["Baseline_Hit@3"].mean(),
        evaluation_results["Baseline_MRR@3"].mean(),
        evaluation_results["Baseline_Correctness"].mean(),
        evaluation_results["Baseline_Groundedness"].mean(),
    ],
    "Advanced": [
        evaluation_results["Advanced_Hit@3"].mean(),
        evaluation_results["Advanced_MRR@3"].mean(),
        evaluation_results["Advanced_Correctness"].mean(),
        evaluation_results["Advanced_Groundedness"].mean(),
    ],
})
summary_table["Difference"] = summary_table["Advanced"] - summary_table["Baseline"]

display(summary_table.round(3))

chart_data = summary_table.set_index("Metric")[["Baseline", "Advanced"]]
ax = chart_data.plot(kind="bar", figsize=(10, 5), color=["#8A8A8A", "#E4572E"])
ax.set_title("Baseline vs Advanced RAG")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

display(answer_details_df)


# Final Summary

아래 셀은 실제 실행 결과를 바탕으로 Baseline Issue, 적용 기법, Before/After, Evaluation, Lessons Learned를 자동으로 정리합니다.


In [ ]:
from IPython.display import Markdown, display

metric_values = summary_table.set_index("Metric")
hit_base = metric_values.loc["Hit@3", "Baseline"]
hit_adv = metric_values.loc["Hit@3", "Advanced"]
mrr_base = metric_values.loc["MRR@3", "Baseline"]
mrr_adv = metric_values.loc["MRR@3", "Advanced"]
correct_base = metric_values.loc["Correctness (1~5)", "Baseline"]
correct_adv = metric_values.loc["Correctness (1~5)", "Advanced"]
ground_base = metric_values.loc["Groundedness (1~5)", "Baseline"]
ground_adv = metric_values.loc["Groundedness (1~5)", "Advanced"]

improved = evaluation_results[
    (evaluation_results["Advanced_MRR@3"] > evaluation_results["Baseline_MRR@3"])
    | (evaluation_results["Advanced_Correctness"] > evaluation_results["Baseline_Correctness"])
]["ID"].tolist()

worsened = evaluation_results[
    (evaluation_results["Advanced_MRR@3"] < evaluation_results["Baseline_MRR@3"])
    | (evaluation_results["Advanced_Correctness"] < evaluation_results["Baseline_Correctness"])
]["ID"].tolist()

retried_ids = evaluation_results[evaluation_results["Verification_Retry"]]["ID"].tolist()

improved_text = ", ".join(improved) if improved else "뚜렷한 개선 사례 없음"
worsened_text = ", ".join(worsened) if worsened else "악화 사례 없음"
retried_text = ", ".join(retried_ids) if retried_ids else "재검색이 필요하다고 판단된 질문 없음"

final_summary = f"""
## 1. Baseline Issue

- Baseline은 Dense Similarity Top-3만 사용하므로 정확한 법률 용어가 중요한 질문에서 정답 판례를 놓칠 수 있었습니다.
- 1,000자 고정 Chunk는 판결 이유와 번호 문단의 경계를 충분히 반영하지 못했습니다.
- 검색 결과가 부족해도 그대로 답변을 생성하여 근거가 약한 답변이 나올 가능성이 있었습니다.

## 2. Applied Techniques

### Technique 1: 판례 구조 기반 Chunking
- 선택 이유: 번호 문단과 판결 이유가 Chunk 경계에서 잘리는 문제를 줄이기 위해 선택했습니다.
- 기대 효과: 질문에 필요한 법리가 하나의 Chunk에 더 온전하게 포함됩니다.

### Technique 2: BM25 + Dense MMR Hybrid Retrieval + RRF
- 선택 이유: 정확한 법률 용어 검색과 의미 기반 검색을 함께 사용하기 위해 선택했습니다.
- 기대 효과: 표현이 다른 질문과 정확한 판례 용어를 모두 처리하고 중복 검색 결과를 줄입니다.

### Technique 3: Evidence Verification + Retry
- 선택 이유: 검색된 판례가 질문의 핵심 쟁점에 직접 답하지 못하는 경우를 탐지하기 위해 선택했습니다.
- 기대 효과: 부족한 근거로 바로 답하지 않고 법률 중심 검색어로 한 번 더 검색합니다.
- 재검색 발생 질문: {retried_text}

## 3. Before / After Comparison

| 항목 | Baseline | Advanced | 차이 |
|---|---:|---:|---:|
| Retrieval Hit@3 | {hit_base:.3f} | {hit_adv:.3f} | {hit_adv-hit_base:+.3f} |
| Retrieval MRR@3 | {mrr_base:.3f} | {mrr_adv:.3f} | {mrr_adv-mrr_base:+.3f} |
| Answer Correctness | {correct_base:.3f} | {correct_adv:.3f} | {correct_adv-correct_base:+.3f} |
| Answer Groundedness | {ground_base:.3f} | {ground_adv:.3f} | {ground_adv-ground_base:+.3f} |

- 개선된 질문: {improved_text}
- 개선되지 않았거나 악화된 질문: {worsened_text}

## 4. Evaluation

- 동일한 Evaluation Set 6개(E01, E05, E07, E10, E11, E12)를 두 모델에 적용했습니다.
- Retrieval은 정답 판례의 Hit@3와 MRR@3로 평가했습니다.
- Answer Quality는 동일한 LLM Judge가 기준 답안과 검색 근거를 사용해 Correctness와 Groundedness를 1~5점으로 평가했습니다.
- 표본이 6개이므로 결과를 전체 저작권 질문으로 일반화하기에는 한계가 있습니다.

## 5. Lessons Learned

- Hybrid Retrieval은 법률 용어의 정확한 일치와 의미 유사성을 함께 활용할 수 있다는 장점이 있습니다.
- 구조 기반 Chunking은 검색된 문맥에 판결의 핵심 법리를 더 온전하게 남기는 데 도움이 됩니다.
- Verification은 근거 부족을 확인하는 안전장치이지만 LLM 호출이 추가되어 비용과 실행 시간이 증가합니다.
- Query가 잘못 확장되면 재검색 결과가 오히려 나빠질 수 있으므로 악화 사례도 함께 확인해야 합니다.
- 다음 실험에서는 Chunk 크기, BM25와 Dense 후보 수, RRF 설정을 같은 평가 세트로 조정할 수 있습니다.

## LangGraph 제외 확인

- LangGraph import: 없음
- Graph/Node/Edge workflow: 없음
- 일반 Python 함수와 LangChain Runnable만 사용
"""

display(Markdown(final_summary))
